# 特征工程（All）


In [ ]:
import gc, os
import logging
import pickle
import time
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
import numpy as np
import pandas as pd
from tqdm import tqdm
import lightgbm as lgb
from gensim.models import Word2Vec
from sklearn.preprocessing import MinMaxScaler

# ============================================================================
# 🎯 核心配置：选择运行模式（必须与 Recall 阶段保持一致！）
# ============================================================================
# "offline" = 离线验证模式：用于评估排序效果，使用划分好的训练/验证集
#             ⚠️ 数据泄漏防护：特征只使用"历史"数据，不使用验证集标签
# "online"  = 在线提交模式：用于生成最终排序结果，使用全量数据
# ============================================================================
MODE = "offline"  # 👈 修改这里切换模式（需要与 4.recall_all.ipynb 保持一致！）

print("=" * 70)
print(f"🔧 特征工程 - 当前运行模式: {MODE}")
if MODE == "offline":
    print("🔬 离线验证模式: 会严格区分训练/验证用户，防止数据泄漏")
else:
    print("🚀 在线提交模式: 使用全量数据做特征工程")
print("=" * 70)


## df节省内存函数

In [ ]:
# 节省内存的一个函数
# 减少内存
def reduce_mem(df):
    starttime = time.time()
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()
            if pd.isnull(c_min) or pd.isnull(c_max):
                continue
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    end_mem = df.memory_usage().sum() / 1024**2
    print('-- Mem. usage decreased to {:5.2f} Mb ({:.1f}% reduction),time spend:{:2.2f} min'.format(end_mem,
                                                                                                           100*(start_mem-end_mem)/start_mem,
                                                                                                           (time.time()-starttime)/60))
    return df


## 定义数据路径

In [ ]:
# 数据路径（独立项目版：纯相对路径）
project_root = Path.cwd().resolve()
data_path = project_root / 'data' / 'raw' / 'news_recommendation'
save_path = project_root / 'data' / 'processed' / 'temp_results'
output_path = project_root / 'outputs'

for path in [data_path, save_path, output_path]:
    path.mkdir(parents=True, exist_ok=True)

print(f"📁 项目根目录: {project_root}")
print(f"📁 原始数据路径: {data_path}")
print(f"📁 中间结果路径: {save_path}")
print(f"📁 导出结果路径: {output_path}")


## 数据读取

### 训练和验证集的划分

划分训练和验证集的原因是为了在线下验证模型参数的好坏，为了完全模拟测试集，我们这里就在训练集中抽取部分用户的所有信息来作为验证集。提前做训练验证集划分的好处就是可以分解制作排序特征时的压力，一次性做整个数据集的排序特征可能时间会比较长。

## ⚠️ 数据泄漏防护设计

### 数据流架构（离线验证模式）

```
原始训练集 (train_click_log.csv)
    │
    ├── 训练用户 (click_trn)
    │       ├── 历史点击 (click_trn_hist)  ─→ 用于构造特征 ✅
    │       └── 最后一次点击 (click_trn_last)  ─→ 仅用于打标签 ⚠️
    │
    └── 验证用户 (click_val + val_ans)
            ├── 历史点击 (click_val_hist = click_val)  ─→ 用于构造特征 ✅
            └── 最后一次点击 (click_val_last = val_ans)  ─→ 仅用于评估 ⚠️
```

### 关键防护点

| 环节 | 使用的数据 | ⚠️ 禁止使用 |
|------|-----------|------------|
| 计算用户活跃度特征 | click_trn_hist + click_val_hist | click_trn_last, val_ans |
| 计算文章热度特征 | click_trn_hist + click_val_hist | click_trn_last, val_ans |
| 训练 Word2Vec | click_trn_hist + click_val_hist | click_trn_last, val_ans |
| 构造交互特征 | 用户各自的历史点击 | 用户各自的最后一次点击 |
| 打标签 | click_trn_last, val_ans | - |

### 与 Recall 阶段的一致性

⚠️ **重要**：特征工程必须与 Recall 阶段使用相同的数据划分！

- 直接读取 Recall 保存的 `click_trn.csv`, `click_val.csv`, `val_ans.csv`
- 确保训练用户/验证用户的划分完全一致
- MODE 设置必须与 Recall 阶段保持一致

In [ ]:
# ============================================================================
# 📊 数据读取函数
# ============================================================================
# ⚠️ 关键设计：
#   - 离线模式：直接读取 Recall 阶段保存的 click_trn, click_val, val_ans
#   - 在线模式：训练集 + 测试集全部数据
# ============================================================================

def get_all_click_df(data_path, mode="offline"):
    """
    读取原始点击数据
    
    参数:
        data_path: 数据路径
        mode: 运行模式
            - "offline": 离线验证模式，只使用训练集
            - "online": 在线提交模式，合并训练集和测试集
    
    返回:
        all_click: 点击数据DataFrame
    """
    if mode == "offline":
        all_click = pd.read_csv(data_path / 'train_click_log.csv')
    else:
        trn_click = pd.read_csv(data_path / 'train_click_log.csv')
        tst_click = pd.read_csv(data_path / 'testA_click_log.csv')
        all_click = pd.concat([trn_click, tst_click]).reset_index(drop=True)
    
    all_click = all_click.drop_duplicates((['user_id', 'click_article_id', 'click_timestamp']))
    return all_click

In [ ]:
# 采样数据（仅用于原始数据参考，实际使用 Recall 保存的划分）
all_click_df = get_all_click_df(data_path, mode=MODE)

这部分是我加的

In [ ]:
# ============================================================================
# 🔬 离线模式：训练集和验证集划分（直接复用 Recall 阶段的划分！）
# ============================================================================
# ⚠️ 重要说明：
#   为了保证与 Recall 阶段完全一致，特征工程直接读取 Recall 保存的文件，
#   而不是重新划分。这样可以确保：
#   1. 训练用户 vs 验证用户的划分完全一致
#   2. 验证用户的历史点击 vs 最后一次点击（标签）划分一致
# ============================================================================

def trn_val_split(all_click_df, sample_user_nums):
    """
    将训练集划分为：训练用户 + 验证用户
    （⚠️ 仅在需要重新划分时使用，一般直接读取 Recall 阶段保存的文件）
    """
    all_click = all_click_df
    all_user_ids = all_click.user_id.unique()
    
    sample_user_ids = np.random.choice(all_user_ids, size=sample_user_nums, replace=False) 
    
    click_val = all_click[all_click['user_id'].isin(sample_user_ids)]
    click_trn = all_click[~all_click['user_id'].isin(sample_user_ids)]
    
    # 将验证用户的最后一次点击提取出来作为 ground truth
    click_val = click_val.sort_values(['user_id', 'click_timestamp'])
    val_ans = click_val.groupby('user_id').tail(1)
    
    # 验证用户去掉最后一次点击，剩下的作为历史
    click_val = click_val.groupby('user_id').apply(lambda x: x[:-1]).reset_index(drop=True)
    
    # 去除只有一条点击的用户
    val_ans = val_ans[val_ans.user_id.isin(click_val.user_id.unique())]
    click_val = click_val[click_val.user_id.isin(val_ans.user_id.unique())]
    
    return click_trn, click_val, val_ans

### 获取历史点击和最后一次点击，这里和前面对单条数据的处理不太一样，前面是直接删掉单条数据，这里是将单条数据作为指标，也作为最后一次点击。

In [ ]:
# ============================================================================
# 📊 获取历史点击和最后一次点击
# ============================================================================
# ⚠️ 数据泄漏防护：
#   - 历史点击（click_hist_df）：用于构造特征
#   - 最后一次点击（click_last_df）：用于打标签（Label），不能用于构造特征！
# ============================================================================

def get_hist_and_last_click(all_click):
    """
    获取用户的历史点击和最后一次点击
    
    返回:
        click_hist_df: 历史点击（用于构造特征）
        click_last_df: 最后一次点击（用于打标签，⚠️不能泄漏到特征中！）
    """
    all_click = all_click.copy()
    
    # ⚠️ 修复 float16 不支持排序的问题：将 click_timestamp 转为 float32
    if all_click['click_timestamp'].dtype == np.float16:
        all_click['click_timestamp'] = all_click['click_timestamp'].astype(np.float32)
    
    all_click = all_click.sort_values(by=['user_id', 'click_timestamp'])
    click_last_df = all_click.groupby('user_id').tail(1)

    # 如果用户只有一个点击，hist为空会导致训练时用户不可见，此时默认泄露一下
    def hist_func(user_df):
        if len(user_df) == 1:
            return user_df
        else:
            return user_df[:-1]

    click_hist_df = all_click.groupby('user_id').apply(hist_func).reset_index(drop=True)

    return click_hist_df, click_last_df

### 读取训练、验证及测试集（原始的）,返回的是按用户分组后的训练集和验证集

In [ ]:
# ============================================================================
# 📊 读取训练、验证及测试集（All 协议）
# ============================================================================
# 离线模式：
#   - 读取 Recall All 阶段保存的 click_hist_all / click_last_all / rank_user_split_all
#   - 先得到全用户 leave-last-out 数据，再按用户切分 rerank train/val
# 在线模式：
#   - 保持原逻辑，读取全量训练集与测试集
# ============================================================================

def get_trn_val_tst_data(data_path, save_path, mode="offline"):
    if mode == "offline":
        print("📂 [离线模式] 读取 Recall All 阶段保存的数据...")
        click_hist_all = pd.read_csv(save_path / 'click_hist_all.csv')
        click_last_all = pd.read_csv(save_path / 'click_last_all.csv')
        rank_user_split = pickle.load(open(save_path / 'rank_user_split_all.pkl', 'rb'))
        click_hist_all = reduce_mem(click_hist_all)
        click_last_all = reduce_mem(click_last_all)

        print(f"   ✅ leave-last-out 用户数: {click_hist_all['user_id'].nunique()}")
        print(f"   ✅ target 数量: {len(click_last_all)}")
        print(f"   ✅ rerank 训练用户数: {len(rank_user_split['train_users'])}")
        print(f"   ✅ rerank 验证用户数: {len(rank_user_split['val_users'])}")

        click_tst = pd.read_csv(data_path / 'testA_click_log.csv')
        click_tst = reduce_mem(click_tst)
        print(f"   ✅ 测试集用户数: {click_tst['user_id'].nunique()}")

        return click_hist_all, click_last_all, click_tst, rank_user_split
    else:
        print("📂 [在线模式] 读取全量训练数据...")
        click_trn = pd.read_csv(data_path / 'train_click_log.csv')
        click_trn = reduce_mem(click_trn)
        click_tst = pd.read_csv(data_path / 'testA_click_log.csv')
        click_tst = reduce_mem(click_tst)
        print(f"   ✅ 训练数据量: {len(click_trn)}")
        print(f"   ✅ 测试集用户数: {click_tst['user_id'].nunique()}")
        return click_trn, None, click_tst, None


### 读取召回列表

In [ ]:
# ============================================================================
# 📊 读取召回列表（All 协议）
# ============================================================================

def get_recall_list(save_path, mode="offline", single_recall_model=None, multi_recall=True):
    """读取召回列表，返回格式: {user_id: [(item_id, score), ...]}"""
    if multi_recall:
        recall_path = save_path / ('offline_all_final_recall_items_dict.pkl' if mode == 'offline' else 'online_all_final_recall_items_dict.pkl')
        print(f"📂 读取多路召回列表: {recall_path}")
        return pickle.load(open(recall_path, 'rb'))

    if single_recall_model == 'i2i_itemcf':
        return pickle.load(open(save_path / 'itemcf_recall_dict_all.pkl', 'rb'))
    elif single_recall_model == 'i2i_emb_itemcf':
        return pickle.load(open(save_path / 'embedding_sim_item_recall_all.pkl', 'rb'))
    elif single_recall_model == 'cold_start':
        return pickle.load(open(save_path / 'cold_start_recall_all.pkl', 'rb'))
    elif single_recall_model in {'user_cf', 'youtubednn'}:
        raise ValueError('chapter_5_projects_all 主链路已移除 YouTubeDNN 相关召回，请改用 itemcf / embedding / cold_start')
    else:
        raise ValueError(f'unknown single_recall_model: {single_recall_model}')


### 读取各种Embedding

#### Word2Vec训练及gensim的使用

Word2Vec主要思想是：一个词的上下文可以很好的表达出词的语义。通过无监督学习产生词向量的方式。word2vec中有两个非常经典的模型：skip-gram和cbow。

- skip-gram：已知中心词预测周围词。
- cbow：已知周围词预测中心词。

在使用gensim训练word2vec的时候，有几个比较重要的参数
- size: 表示词向量的维度。
- window：决定了目标词会与多远距离的上下文产生关系。
- sg: 如果是0，则是CBOW模型，是1则是Skip-Gram模型。
- workers: 表示训练时候的线程数量
- min_count: 设置最小的
- iter: 训练时遍历整个数据集的次数

**注意**
1. 训练的时候输入的语料库一定要是字符组成的二维数组，如：[['北', '京', '你', '好'], ['上', '海', '你', '好']]
2. 使用模型的时候有一些默认值，可以通过在Jupyter里面通过`Word2Vec??`查看


下面是个简单的测试样例：

In [ ]:
from gensim.models import Word2Vec
docs = [['30760', '157507'],
       ['289197', '63746'],
       ['36162', '168401'],
       ['50644', '36162']]
w2v = Word2Vec(docs, vector_size=12, sg=1, window=2, seed=2020, workers=2, min_count=1, epochs=1)

# 查看'30760'表示的词向量
w2v.wv['30760']


skip-gram和cbow的详细原理可以参考下面的博客：
- [word2vec原理(一) CBOW与Skip-Gram模型基础](https://www.cnblogs.com/pinard/p/7160330.html)   
- [word2vec原理(二) 基于Hierarchical Softmax的模型](https://www.cnblogs.com/pinard/p/7160330.html)   
- [word2vec原理(三) 基于Negative Sampling的模型](https://www.cnblogs.com/pinard/p/7249903.html)

In [ ]:
def trian_item_word2vec(click_df, embed_size=64, save_name='item_w2v_emb_all.pkl', split_char=' '):
    click_df = click_df.sort_values('click_timestamp')

    # 只有转换成字符串才可以进行训练
    click_df['click_article_id'] = click_df['click_article_id'].astype(str)

    # 转换成句子的形式
    docs = click_df.groupby(['user_id'])['click_article_id'].apply(lambda x: list(x)).reset_index()
    docs = docs['click_article_id'].values.tolist()

    logging.basicConfig(format='%(asctime)s:%(levelname)s:%(message)s', level=logging.INFO)
    model = Word2Vec(docs, vector_size=16, sg=1, window=5, seed=2020, workers=24, min_count=1, epochs=1)

    item_w2v_emb_dict = {k: model.wv[k] for k in click_df['click_article_id']}
    pickle.dump(item_w2v_emb_dict, open(save_path / save_name, 'wb'))

    return item_w2v_emb_dict


In [ ]:
# 可以通过字典查询对应的 item embedding
def get_embedding(save_path, all_click_df):
    item_content_emb_dict = None
    item_w2v_emb_dict = None

    if os.path.exists(save_path / 'item_content_emb_all.pkl'):
        item_content_emb_dict = pickle.load(open(save_path / 'item_content_emb_all.pkl', 'rb'))
    else:
        print('item_content_emb_all.pkl 文件不存在...')

    if os.path.exists(save_path / 'item_w2v_emb_all.pkl'):
        item_w2v_emb_dict = pickle.load(open(save_path / 'item_w2v_emb_all.pkl', 'rb'))
    else:
        item_w2v_emb_dict = trian_item_word2vec(all_click_df)

    return item_content_emb_dict, item_w2v_emb_dict


### 读取文章信息

In [ ]:
def get_article_info_df():
    article_info_df = pd.read_csv(data_path / 'articles.csv')
    article_info_df = reduce_mem(article_info_df)
    
    return article_info_df


### 读取数据

In [ ]:
# ============================================================================
# 📊 读取数据（根据 MODE 自动选择）
# ============================================================================
if MODE == "offline":
    click_hist_all, click_last_all, click_tst, rank_user_split = get_trn_val_tst_data(data_path, save_path, mode=MODE)
    click_trn = None
    click_val = None
    val_ans = None
else:
    click_trn, click_val, click_tst, val_ans = get_trn_val_tst_data(data_path, save_path, mode=MODE)
    click_hist_all = None
    click_last_all = None
    rank_user_split = None

print("\n" + "=" * 70)
print("📊 数据加载完成")
print("=" * 70)


In [ ]:
# ============================================================================
# 📊 获取历史点击和最后一次点击（All 协议）
# ============================================================================
# 离线模式：
#   1. Recall 层已经基于全用户 leave-last-out 生成 click_hist_all / click_last_all
#   2. 这里再按 rank_user_split_all.pkl 将用户切成 rerank train / val
# 在线模式：
#   保持原逻辑，对训练集做 leave-last-out，测试集全部作为历史
# ============================================================================

if MODE == "offline":
    train_users = set(rank_user_split['train_users'])
    val_users = set(rank_user_split['val_users'])

    click_trn_hist = click_hist_all[click_hist_all['user_id'].isin(train_users)].reset_index(drop=True)
    click_val_hist = click_hist_all[click_hist_all['user_id'].isin(val_users)].reset_index(drop=True)
    click_trn_last = click_last_all[click_last_all['user_id'].isin(train_users)].reset_index(drop=True)
    click_val_last = click_last_all[click_last_all['user_id'].isin(val_users)].reset_index(drop=True)
    click_tst_hist = click_tst

    print(f"📊 rerank 训练用户历史点击数: {len(click_trn_hist)}, target 数: {len(click_trn_last)}")
    print(f"📊 rerank 验证用户历史点击数: {len(click_val_hist)}, target 数: {len(click_val_last)}")
else:
    click_trn_hist, click_trn_last = get_hist_and_last_click(click_trn)
    print(f"📊 训练用户历史点击数: {len(click_trn_hist)}, 最后一次点击数: {len(click_trn_last)}")

    click_val_hist, click_val_last = None, None
    click_tst_hist = click_tst

print(f"📊 测试集用户点击数: {len(click_tst_hist)}")


## 对训练数据做负采样

通过召回我们将数据转换成三元组的形式（user1, item1, label）的形式，观察发现正负样本差距极度不平衡，我们可以先对负样本进行下采样，下采样的目的一方面缓解了正负样本比例的问题，另一方面也减小了我们做排序特征的压力，我们在做负采样的时候又有哪些东西是需要注意的呢？

1. 只对负样本进行下采样(如果有比较好的正样本扩充的方法其实也是可以考虑的)
2. 负采样之后，保证所有的用户和文章仍然出现在采样之后的数据中
3. 下采样的比例可以根据实际情况人为的控制
4. 做完负采样之后，更新此时新的用户召回文章列表，因为后续做特征的时候可能用到相对位置的信息。

其实负采样也可以留在后面做完特征在进行，这里由于做排序特征太慢了，所以把负采样的环节提到前面了。

In [ ]:
# 将召回列表转换成df的形式
def recall_dict_2_df(recall_list_dict):
    df_row_list = [] # [user, item, score]
    for user, recall_list in tqdm(recall_list_dict.items(), disable=not logger.isEnabledFor(logging.DEBUG)):
        # 新增：兼容字典格式（防止上一步保存错格式）
        if isinstance(recall_list, dict):
            recall_list = recall_list.items()
            
        for item, score in recall_list:
            df_row_list.append([user, item, score])
    
    col_names = ['user_id', 'sim_item', 'score']
    recall_list_df = pd.DataFrame(df_row_list, columns=col_names)
    
    return recall_list_df


In [ ]:
# 负采样函数，这里可以控制负采样时的比例, 这里给了一个默认的值
def neg_sample_recall_data(recall_items_df, sample_rate=0.001):
    #先按照label拆分吗正负样本
    pos_data = recall_items_df[recall_items_df['label'] == 1]
    neg_data = recall_items_df[recall_items_df['label'] == 0]
    
    #打印正负样本比例
    print('pos_data_num:', len(pos_data), 'neg_data_num:', len(neg_data), 'pos/neg:', len(pos_data)/len(neg_data))
    
    # 分组采样函数
    def neg_sample_func(group_df):
        neg_num = len(group_df)
        sample_num = max(int(neg_num * sample_rate), 1) # 保证最少有一个
        sample_num = min(sample_num, 5) # 保证最多不超过5个，这里可以根据实际情况进行选择
        return group_df.sample(n=sample_num, replace=True)
    
    # 对用户进行负采样，保证所有用户都在采样后的数据中
    neg_data_user_sample = neg_data.groupby('user_id', group_keys=False).apply(neg_sample_func)
    # 对文章进行负采样，保证所有文章都在采样后的数据中
    neg_data_item_sample = neg_data.groupby('sim_item', group_keys=False).apply(neg_sample_func)
    
    # 将上述两种情况下的采样数据合并    
    neg_data_new = pd.concat([neg_data_user_sample, neg_data_item_sample]).reset_index(drop=True)
    # 由于上述两个操作是分开的，可能将两个相同的数据给重复选择了，所以需要对合并后的数据进行去重
    neg_data_new = neg_data_new.sort_values(['user_id', 'score']).drop_duplicates(['user_id', 'sim_item'], keep='last')
    
    # 将正样本数据合并
    data_new = pd.concat([pos_data, neg_data_new], ignore_index=True)
    
    return data_new


In [ ]:
# 召回数据打标签
def get_rank_label_df(recall_list_df, label_df, is_test=False):
    # 测试集是没有标签了，为了后面代码同一一些，这里直接给一个负数替代
    if is_test:
        recall_list_df['label'] = -1
        return recall_list_df
    
    label_df = label_df.rename(columns={'click_article_id': 'sim_item'})#统一名称

    recall_list_df_ = recall_list_df.merge(label_df[['user_id', 'sim_item', 'click_timestamp']], \
                                               how='left', on=['user_id', 'sim_item'])
#有 timestamp → 说明 (user, item) 在 label 表中存在，说明点击过
    recall_list_df_['label'] = recall_list_df_['click_timestamp'].apply(lambda x: 0.0 if np.isnan(x) else 1.0)
                                                         
    del recall_list_df_['click_timestamp']
    
    return recall_list_df_


##从召回列表中筛选训练用户，验证用户和测试用户，并且根据最后一次点击打标签并且进行负采样的过程。

In [ ]:
# ============================================================================
# 📊 从召回列表中筛选用户并打标签（⚠️ 重要的数据流控制点！）
# ============================================================================
# 
# 数据流说明：
#   离线模式：
#     - 训练用户：用 click_trn_last（最后一次点击）打标签
#     - 验证用户：用 click_val_last = val_ans 打标签
#     - 测试用户：标签为 -1（实际提交时不需要标签）
#   
#   在线模式：
#     - 训练用户：用 click_trn_last 打标签
#     - 测试用户：标签为 -1
# ============================================================================

def get_user_recall_item_label_df(click_trn_hist, click_val_hist, click_tst_hist,
                                   click_trn_last, click_val_last, recall_list_df,
                                   mode="offline"):
    """
    从召回列表中筛选训练用户、验证用户和测试用户，并打标签和负采样
    
    参数:
        click_trn_hist: 训练用户的历史点击（不含最后一次）
        click_val_hist: 验证用户的历史点击（不含最后一次）
        click_tst_hist: 测试用户的点击（全部，因为要预测下一次）
        click_trn_last: 训练用户的最后一次点击（用于打 Label）
        click_val_last: 验证用户的最后一次点击（用于打 Label）
        recall_list_df: 召回列表 DataFrame
        mode: 运行模式
    
    返回:
        trn_user_item_label_df: 训练数据（含负采样）
        val_user_item_label_df: 验证数据（不做负采样，用于完整评估）
        tst_user_item_label_df: 测试数据（标签为 -1）
    """
    print("\n" + "=" * 70)
    print("📊 处理召回数据：筛选用户、打标签、负采样")
    print("=" * 70)
    
    # ─────────────────────────────────────────────────────────────────────
    # 1. 训练用户：筛选 + 打标签 + 负采样
    # ─────────────────────────────────────────────────────────────────────
    trn_users = click_trn_hist['user_id'].unique()
    trn_user_items_df = recall_list_df[recall_list_df['user_id'].isin(trn_users)]
    print(f"📊 训练用户召回候选数: {len(trn_user_items_df)}")
    
    # 打标签（用 click_trn_last）
    trn_user_item_label_df = get_rank_label_df(trn_user_items_df, click_trn_last, is_test=False)
    
    # 负采样
    trn_user_item_label_df = neg_sample_recall_data(trn_user_item_label_df)
    print(f"📊 训练用户负采样后数据量: {len(trn_user_item_label_df)}")
    
    # ─────────────────────────────────────────────────────────────────────
    # 2. 验证用户（仅离线模式）：筛选 + 打标签（不做负采样，保证完整评估）
    # ─────────────────────────────────────────────────────────────────────
    if mode == "offline" and click_val_hist is not None:
        val_users = click_val_hist['user_id'].unique()
        val_user_items_df = recall_list_df[recall_list_df['user_id'].isin(val_users)]
        print(f"📊 验证用户召回候选数: {len(val_user_items_df)}")
        
        # 打标签（用 click_val_last = val_ans）
        val_user_item_label_df = get_rank_label_df(val_user_items_df, click_val_last, is_test=False)
        # ⚠️ 验证集不做负采样，保证评估时能看到全部候选
        print(f"📊 验证用户数据量（不负采样）: {len(val_user_item_label_df)}")
    else:
        val_user_item_label_df = None
        
    # ─────────────────────────────────────────────────────────────────────
    # 3. 测试用户：筛选 + 打 -1 标签（无真实标签）
    # ─────────────────────────────────────────────────────────────────────
    tst_users = click_tst_hist['user_id'].unique()
    tst_user_items_df = recall_list_df[recall_list_df['user_id'].isin(tst_users)]
    print(f"📊 测试用户召回候选数: {len(tst_user_items_df)}")
    
    tst_user_item_label_df = get_rank_label_df(tst_user_items_df, None, is_test=True)
    
    print("=" * 70)
    
    return trn_user_item_label_df, val_user_item_label_df, tst_user_item_label_df

In [ ]:
# 读取召回列表
recall_list_dict = get_recall_list(save_path, mode=MODE, single_recall_model=None, multi_recall=True)
# 将召回数据转换成df
recall_list_df = recall_dict_2_df(recall_list_dict)


In [ ]:
# ============================================================================
# 📊 执行打标签和负采样
# ============================================================================

trn_user_item_label_df, val_user_item_label_df, tst_user_item_label_df = get_user_recall_item_label_df(
    click_trn_hist,
    click_val_hist,
    click_tst_hist,
    click_trn_last,
    click_val_last,
    recall_list_df,
    mode=MODE,
)


def report_label_stats(df, name):
    if df is None or len(df) == 0:
        print(f"📊 {name}: 空")
        return
    user_cnt = df['user_id'].nunique()
    pos_user_cnt = int(df.groupby('user_id')['label'].max().sum()) if 'label' in df.columns else 0
    pos_row_cnt = int((df['label'] == 1).sum()) if 'label' in df.columns else 0
    print(f"📊 {name} 用户数: {user_cnt}")
    print(f"   有正样本用户数: {pos_user_cnt}")
    print(f"   正样本行数: {pos_row_cnt}")
    print(f"   平均候选数: {len(df) / user_cnt:.2f}")


report_label_stats(trn_user_item_label_df, '训练集')
if val_user_item_label_df is not None:
    report_label_stats(val_user_item_label_df, '验证集')
report_label_stats(tst_user_item_label_df, '测试集')


In [ ]:
trn_user_item_label_df.label


## 将召回数据转换成字典

In [ ]:
# 将最终的召回的df数据转换成字典的形式做排序特征
def make_tuple_func(group_df):
    row_data = []
    for name, row_df in group_df.iterrows():
        row_data.append((row_df['sim_item'], row_df['score'], row_df['label']))
    
    return row_data


In [ ]:
trn_user_item_label_tuples_dict = trn_user_item_label_df.groupby('user_id').apply(make_tuple_func).to_dict()

if val_user_item_label_df is not None:
    val_user_item_label_tuples_dict = val_user_item_label_df.groupby('user_id').apply(make_tuple_func).to_dict()
else:
    val_user_item_label_tuples_dict = None
    
tst_user_item_label_tuples_dict = tst_user_item_label_df.groupby('user_id').apply(make_tuple_func).to_dict()


## 用户历史行为相关特征

对于每个用户召回的每个商品， 做特征。 具体步骤如下：
* 对于每个用户， 获取最后点击的N个商品的item_id， 
    * 对于该用户的每个召回商品， 计算与上面最后N次点击商品的相似度的和(最大， 最小，均值)， 时间差特征，相似性特征，字数差特征，与该用户的相似性特征

In [ ]:
# 下面基于data做历史相关的特征
def create_feature(users_id, recall_list, click_hist_df,  articles_info, articles_emb, user_emb=None, N=1):
    """
    基于用户的历史行为做相关特征
    :param users_id: 用户id
    :param recall_list: 对于每个用户召回的候选文章列表
    :param click_hist_df: 用户的历史点击信息
    :param articles_info: 文章信息
    :param articles_emb: 文章的embedding向量, 这个可以用item_content_emb, item_w2v_emb, item_youtube_emb
    :param user_emb: 用户的embedding向量， 这个是user_youtube_emb, 如果没有也可以不用， 但要注意如果要用的话， articles_emb就要用item_youtube_emb的形式， 这样维度才一样
    :param N: 最近的N次点击  由于testA日志里面很多用户只存在一次历史点击， 所以为了不产生空值，默认是1
    """
    
    # 建立一个二维列表保存结果， 后面要转成DataFrame
    all_user_feas = []
    i = 0
    for user_id in tqdm(users_id, disable=not logger.isEnabledFor(logging.DEBUG)):
        # 该用户的最后N次点击
        hist_user_items = click_hist_df[click_hist_df['user_id']==user_id]['click_article_id'][-N:]
        
        # 遍历该用户的召回列表
        for rank, (article_id, score, label) in enumerate(recall_list[user_id]):
            # 该文章建立时间, 字数
            a_create_time = articles_info[articles_info['article_id']==article_id]['created_at_ts'].values[0]
            a_words_count = articles_info[articles_info['article_id']==article_id]['words_count'].values[0]
            single_user_fea = [user_id, article_id]
            # 计算与最后点击的商品的相似度的和， 最大值和最小值， 均值
            sim_fea = []
            time_fea = []
            word_fea = []
            # 遍历用户的最后N次点击文章
            for hist_item in hist_user_items:
                b_create_time = articles_info[articles_info['article_id']==hist_item]['created_at_ts'].values[0]
                b_words_count = articles_info[articles_info['article_id']==hist_item]['words_count'].values[0]
                
                sim_fea.append(np.dot(articles_emb[hist_item], articles_emb[article_id]))
                time_fea.append(abs(a_create_time-b_create_time))
                word_fea.append(abs(a_words_count-b_words_count))
                
            single_user_fea.extend(sim_fea)      # 相似性特征
            single_user_fea.extend(time_fea)    # 时间差特征
            single_user_fea.extend(word_fea)    # 字数差特征
            single_user_fea.extend([max(sim_fea), min(sim_fea), sum(sim_fea), sum(sim_fea) / len(sim_fea)])  # 相似性的统计特征
            
            if user_emb:  # 如果用户向量有的话， 这里计算该召回文章与用户的相似性特征 
                single_user_fea.append(np.dot(user_emb[user_id], articles_emb[article_id]))
                
            single_user_fea.extend([score, rank, label])    
            # 加入到总的表中
            all_user_feas.append(single_user_fea)
    
    # 定义列名
    id_cols = ['user_id', 'click_article_id']
    sim_cols = ['sim' + str(i) for i in range(N)]
    time_cols = ['time_diff' + str(i) for i in range(N)]
    word_cols = ['word_diff' + str(i) for i in range(N)]
    sat_cols = ['sim_max', 'sim_min', 'sim_sum', 'sim_mean']
    user_item_sim_cols = ['user_item_sim'] if user_emb else []
    user_score_rank_label = ['score', 'rank', 'label']
    cols = id_cols + sim_cols + time_cols + word_cols + sat_cols + user_item_sim_cols + user_score_rank_label
            
    # 转成DataFrame
    df = pd.DataFrame( all_user_feas, columns=cols)
    
    return df


In [ ]:
# ============================================================================
# 📊 加载 Embedding（⚠️ 确保 Embedding 训练也不包含未来信息）
# ============================================================================

article_info_df = get_article_info_df()

if MODE == "offline":
    all_click_for_emb = click_hist_all.copy()
else:
    all_click_for_emb = click_trn_hist.copy()

item_content_emb_dict, item_w2v_emb_dict = get_embedding(save_path, all_click_for_emb)

print(f"📊 Embedding 加载完成")


In [ ]:
# ============================================================================
# 📊 构造用户-文章交互特征（⚠️ 使用正确的历史数据！）
# ============================================================================
# 
# 特征构造时使用的历史数据：
#   - 训练用户：click_trn_hist（不含最后一次点击）
#   - 验证用户：click_val_hist = click_val（不含最后一次点击）
#   - 测试用户：click_tst_hist = click_tst（全部历史）
# ============================================================================

print("\n" + "=" * 70)
print("📊 构造用户-文章交互特征（基于历史点击）")
print("=" * 70)

# 训练用户特征
print("📊 处理训练用户特征...")
trn_user_item_feats_df = create_feature(
    trn_user_item_label_tuples_dict.keys(), 
    trn_user_item_label_tuples_dict, 
    click_trn_hist,  # ⚠️ 使用历史点击，不含最后一次
    article_info_df, 
    item_content_emb_dict
)

# 验证用户特征（仅离线模式）
if val_user_item_label_tuples_dict is not None:
    print("📊 处理验证用户特征...")
    val_user_item_feats_df = create_feature(
        val_user_item_label_tuples_dict.keys(), 
        val_user_item_label_tuples_dict, 
        click_val_hist,  # ⚠️ 使用历史点击，不含最后一次
        article_info_df, 
        item_content_emb_dict
    )
else:
    val_user_item_feats_df = None

# 测试用户特征
print("📊 处理测试用户特征...")
tst_user_item_feats_df = create_feature(
    tst_user_item_label_tuples_dict.keys(), 
    tst_user_item_label_tuples_dict, 
    click_tst_hist,  # 测试集全部作为历史
    article_info_df, 
    item_content_emb_dict
)

print("=" * 70)
print("📊 用户-文章交互特征构造完成")
print("=" * 70)

In [ ]:
# ============================================================================
# 📊 保存用户-文章交互特征
# ============================================================================

print("📂 保存用户-文章交互特征...")

trn_user_item_feats_df.to_csv(save_path / 'trn_user_item_feats_df_all.csv', index=False)
print(f"   ✅ 训练特征: {save_path / 'trn_user_item_feats_df_all.csv'}")

if val_user_item_feats_df is not None:
    val_user_item_feats_df.to_csv(save_path / 'val_user_item_feats_df_all.csv', index=False)
    print(f"   ✅ 验证特征: {save_path / 'val_user_item_feats_df_all.csv'}")

tst_user_item_feats_df.to_csv(save_path / 'tst_user_item_feats_df_all.csv', index=False)
print(f"   ✅ 测试特征: {save_path / 'tst_user_item_feats_df_all.csv'}")


## 用户和文章特征

### 用户相关特征

这一块，正式进行特征工程，既要拼接上已有的特征， 也会做更多的特征出来，我们来梳理一下已有的特征和可构造特征：
1. 文章自身的特征， 文章字数，文章创建时间， 文章的embedding （articles表中)
2. 用户点击环境特征， 那些设备的特征(这个在df中)
3. 对于用户和商品还可以构造的特征：
    * 基于用户的点击文章次数和点击时间构造可以表现用户活跃度的特征
    * 基于文章被点击次数和时间构造可以反映文章热度的特征
    * 用户的时间统计特征： 根据其点击的历史文章列表的点击时间和文章的创建时间做统计特征，比如求均值， 这个可以反映用户对于文章时效的偏好
    * 用户的主题爱好特征， 对于用户点击的历史文章主题进行一个统计， 然后对于当前文章看看是否属于用户已经点击过的主题
    * 用户的字数爱好特征， 对于用户点击的历史文章的字数统计， 求一个均值

In [ ]:
click_tst.head()


In [ ]:
# ============================================================================
# 📊 此单元格已废弃，统计特征构造逻辑已移至下一个单元格
# ============================================================================
# 原因：需要严格控制用于统计的数据，防止数据泄漏
# 新逻辑见下一个单元格
# ============================================================================

In [ ]:
# ============================================================================
# 📊 构造用户/文章统计特征（All 协议）
# ============================================================================

articles = pd.read_csv(data_path / 'articles.csv')
articles = reduce_mem(articles)

if MODE == "offline":
    print("📊 [离线模式] 构造统计特征，使用数据：")
    print("   - leave-last-out 历史点击（click_hist_all）")
    print("   - 测试集全部点击（click_tst）")
    all_data_for_stats = pd.concat([click_hist_all, click_tst]).reset_index(drop=True)
else:
    print("📊 [在线模式] 构造统计特征，使用数据：")
    print("   - 训练用户历史点击（click_trn_hist）")
    print("   - 测试集全部点击（click_tst）")
    all_data_for_stats = pd.concat([click_trn_hist, click_tst]).reset_index(drop=True)

all_data = all_data_for_stats.merge(articles, left_on='click_article_id', right_on='article_id')

print(f"\n📊 用于统计特征的数据量: {len(all_data)}")
print(f"📊 涉及用户数: {all_data['user_id'].nunique()}")
print(f"📊 涉及文章数: {all_data['click_article_id'].nunique()}")


In [ ]:
all_data.shape


#### 分析一下点击时间和点击文章的次数，区分用户活跃度
如果某个用户点击文章之间的时间间隔比较小， 同时点击的文章次数很多的话， 那么我们认为这种用户一般就是活跃用户, 当然衡量用户活跃度的方式可能多种多样， 这里我们只提供其中一种，我们写一个函数， 得到可以衡量用户活跃度的特征，逻辑如下：
1. 首先根据用户user_id分组， 对于每个用户，计算点击文章的次数， 两两点击文章时间间隔的均值
2. 把点击次数取倒数和时间间隔的均值统一归一化，然后两者相加合并，该值越小， 说明用户越活跃
3. 注意， 上面两两点击文章的时间间隔均值， 会出现如果用户只点击了一次的情况，这时候时间间隔均值那里会出现空值， 对于这种情况最后特征那里给个大数进行区分

这个的衡量标准就是先把点击的次数取到数然后归一化， 然后点击的时间差归一化， 然后两者相加进行合并， 该值越小， 说明被点击的次数越多， 且间隔时间短。

In [ ]:
# ============================================================================
# 📊 用户活跃度特征
# ============================================================================
# ⚠️ 使用的数据：all_data（只包含历史点击，不含标签）
#    确保无数据泄漏
# ============================================================================

def active_level(all_data, cols):
    """
    制作衡量用户活跃度的特征
    
    逻辑：
    1. 点击次数越多，活跃度越高
    2. 点击时间间隔越短，活跃度越高
    
    参数:
        all_data: 用于统计的数据（⚠️ 必须只包含历史点击，不含标签）
        cols: 用到的特征列
    """
    data = all_data[cols].copy()
    data.sort_values(['user_id', 'click_timestamp'], inplace=True)
    user_act = pd.DataFrame(data.groupby('user_id', as_index=False)[['click_article_id', 'click_timestamp']].\
                            agg({'click_article_id':np.size, 'click_timestamp': {list}}).values, columns=['user_id', 'click_size', 'click_timestamp'])
    
    # 计算时间间隔的均值
    def time_diff_mean(l):
        if len(l) == 1:
            return 1
        else:
            return np.mean([j-i for i, j in list(zip(l[:-1], l[1:]))])
        
    user_act['time_diff_mean'] = user_act['click_timestamp'].apply(lambda x: time_diff_mean(x))
    
    # 点击次数取倒数
    user_act['click_size'] = 1 / user_act['click_size']
    
    # 两者归一化
    user_act['click_size'] = (user_act['click_size'] - user_act['click_size'].min()) / (user_act['click_size'].max() - user_act['click_size'].min())
    user_act['time_diff_mean'] = (user_act['time_diff_mean'] - user_act['time_diff_mean'].min()) / (user_act['time_diff_mean'].max() - user_act['time_diff_mean'].min())     
    user_act['active_level'] = user_act['click_size'] + user_act['time_diff_mean']
    
    user_act['user_id'] = user_act['user_id'].astype('int')
    del user_act['click_timestamp']
    return user_act

In [ ]:
user_act_fea = active_level(all_data, ['user_id', 'click_article_id', 'click_timestamp'])


In [ ]:
user_act_fea.head()


#### 分析一下点击时间和被点击文章的次数， 衡量文章热度特征
和上面同样的思路， 如果一篇文章在很短的时间间隔之内被点击了很多次， 说明文章比较热门，实现的逻辑和上面的基本一致， 只不过这里是按照点击的文章进行分组：
1. 根据文章进行分组， 对于每篇文章的用户， 计算点击的时间间隔
2. 将用户的数量取倒数， 然后用户的数量和时间间隔归一化， 然后相加得到热度特征， 该值越小， 说明被点击的次数越大且时间间隔越短， 文章比较热

当然， 这只是给出一种判断文章热度的一种方法， 这里大家也可以头脑风暴一下

In [ ]:
# ============================================================================
# 📊 文章热度特征
# ============================================================================
# ⚠️ 使用的数据：all_data（只包含历史点击，不含标签）
#    确保无数据泄漏
# ============================================================================

def hot_level(all_data, cols):
    """
    制作衡量文章热度的特征
    
    逻辑：
    1. 被点击次数越多，热度越高
    2. 点击时间间隔越短，热度越高
    
    参数:
        all_data: 用于统计的数据（⚠️ 必须只包含历史点击，不含标签）
        cols: 用到的特征列
    """
    data = all_data[cols].copy()
    data.sort_values(['click_article_id', 'click_timestamp'], inplace=True)
    article_hot = pd.DataFrame(data.groupby('click_article_id', as_index=False)[['user_id', 'click_timestamp']].\
                            agg({'user_id':np.size, 'click_timestamp': {list}}).values, columns=['click_article_id', 'user_num', 'click_timestamp'])
    
    # 计算被点击时间间隔的均值
    def time_diff_mean(l):
        if len(l) == 1:
            return 1
        else:
            return np.mean([j-i for i, j in list(zip(l[:-1], l[1:]))])
        
    article_hot['time_diff_mean'] = article_hot['click_timestamp'].apply(lambda x: time_diff_mean(x))
    
    # 点击次数取倒数
    article_hot['user_num'] = 1 / article_hot['user_num']
    
    # 两者归一化
    article_hot['user_num'] = (article_hot['user_num'] - article_hot['user_num'].min()) / (article_hot['user_num'].max() - article_hot['user_num'].min())
    article_hot['time_diff_mean'] = (article_hot['time_diff_mean'] - article_hot['time_diff_mean'].min()) / (article_hot['time_diff_mean'].max() - article_hot['time_diff_mean'].min())     
    article_hot['hot_level'] = article_hot['user_num'] + article_hot['time_diff_mean']
    
    article_hot['click_article_id'] = article_hot['click_article_id'].astype('int')
    
    del article_hot['click_timestamp']
    
    return article_hot

In [ ]:
article_hot_fea = hot_level(all_data, ['user_id', 'click_article_id', 'click_timestamp'])    


In [ ]:
article_hot_fea.head()


#### 用户的系列习惯
这个基于原来的日志表做一个类似于article的那种DataFrame， 存放用户特有的信息, 主要包括点击习惯， 爱好特征之类的
* 用户的设备习惯， 这里取最常用的设备（众数）
* 用户的时间习惯： 根据其点击过得历史文章的时间来做一个统计（这个感觉最好是把时间戳里的时间特征的h特征提出来，看看用户习惯一天的啥时候点击文章）， 但这里先用转换的时间吧， 求个均值
* 用户的爱好特征， 对于用户点击的历史文章主题进行用户的爱好判别， 更偏向于哪几个主题， 这个最好是multi-hot进行编码， 先试试行不
* 用户文章的字数差特征， 用户的爱好文章的字数习惯

这些就是对用户进行分组， 然后统计即可

#### 用户的设备习惯

In [ ]:
def device_fea(all_data, cols):
    """
    制作用户的设备特征
    :param all_data: 数据集
    :param cols: 用到的特征列
    """
    user_device_info = all_data[cols]
    
    # 用众数来表示每个用户的设备信息
    user_device_info = user_device_info.groupby('user_id').agg(lambda x: x.value_counts().index[0]).reset_index()
    
    return user_device_info


In [ ]:
# 设备特征(这里时间会比较长)
device_cols = ['user_id', 'click_environment', 'click_deviceGroup', 'click_os', 'click_country', 'click_region', 'click_referrer_type']
user_device_info = device_fea(all_data, device_cols)


In [ ]:
user_device_info.head()


#### 用户的时间习惯

In [ ]:
def user_time_hob_fea(all_data, cols):
    """
    制作用户的时间习惯特征
    :param all_data: 数据集
    :param cols: 用到的特征列
    """
    user_time_hob_info = all_data[cols]
    
    # 先把时间戳进行归一化
    mm = MinMaxScaler()
    user_time_hob_info['click_timestamp'] = mm.fit_transform(user_time_hob_info[['click_timestamp']])
    user_time_hob_info['created_at_ts'] = mm.fit_transform(user_time_hob_info[['created_at_ts']])

    user_time_hob_info = user_time_hob_info.groupby('user_id').agg('mean').reset_index()
    
    user_time_hob_info.rename(columns={'click_timestamp': 'user_time_hob1', 'created_at_ts': 'user_time_hob2'}, inplace=True)
    return user_time_hob_info


In [ ]:
user_time_hob_cols = ['user_id', 'click_timestamp', 'created_at_ts']
user_time_hob_info = user_time_hob_fea(all_data, user_time_hob_cols)


#### 用户的主题爱好
这里先把用户点击的文章属于的主题转成一个列表， 后面再总的汇总的时候单独制作一个特征， 就是文章的主题如果属于这里面， 就是1， 否则就是0。

In [ ]:
def user_cat_hob_fea(all_data, cols):
    """
    用户的主题爱好
    :param all_data: 数据集
    :param cols: 用到的特征列
    """
    user_category_hob_info = all_data[cols]
    user_category_hob_info = user_category_hob_info.groupby('user_id').agg({list}).reset_index()
    
    user_cat_hob_info = pd.DataFrame()
    user_cat_hob_info['user_id'] = user_category_hob_info['user_id']
    user_cat_hob_info['cate_list'] = user_category_hob_info['category_id']
    
    return user_cat_hob_info


In [ ]:
user_category_hob_cols = ['user_id', 'category_id']
user_cat_hob_info = user_cat_hob_fea(all_data, user_category_hob_cols)


#### 用户的字数偏好特征

In [ ]:
user_wcou_info = all_data.groupby('user_id')['words_count'].agg('mean').reset_index()
user_wcou_info.rename(columns={'words_count': 'words_hbo'}, inplace=True)


#### 用户的信息特征合并保存

In [ ]:
# 所有表进行合并

user_info = pd.merge(user_act_fea, user_device_info, on='user_id')
user_info = user_info.merge(user_time_hob_info, on='user_id')
user_info = user_info.merge(user_cat_hob_info, on='user_id')
user_info = user_info.merge(user_wcou_info, on='user_id')



In [ ]:
# ============================================================================
# 📂 保存用户特征
# ============================================================================
# ⚠️ 这些特征是基于历史点击数据计算的，不包含标签信息
# ============================================================================

user_info.to_csv(save_path / 'user_info_all.csv', index=False)
print(f"📂 用户特征已保存: {save_path / 'user_info_all.csv'}")
print(f"   用户数: {len(user_info)}")


### 用户特征直接读入
如果前面关于用户的特征工程已经给做完了，后面可以直接读取

In [ ]:
# 把用户信息直接读入进来
user_info = pd.read_csv(save_path / 'user_info_all.csv')


In [ ]:
if os.path.exists(save_path / 'trn_user_item_feats_df_all.csv'):
    trn_user_item_feats_df = pd.read_csv(save_path / 'trn_user_item_feats_df_all.csv')

if os.path.exists(save_path / 'tst_user_item_feats_df_all.csv'):
    tst_user_item_feats_df = pd.read_csv(save_path / 'tst_user_item_feats_df_all.csv')

if os.path.exists(save_path / 'val_user_item_feats_df_all.csv'):
    val_user_item_feats_df = pd.read_csv(save_path / 'val_user_item_feats_df_all.csv')
else:
    val_user_item_feats_df = None


In [ ]:
# 拼上用户特征
# 下面是线下验证的
trn_user_item_feats_df = trn_user_item_feats_df.merge(user_info, on='user_id', how='left')

if val_user_item_feats_df is not None:
    val_user_item_feats_df = val_user_item_feats_df.merge(user_info, on='user_id', how='left')
else:
    val_user_item_feats_df = None
    
tst_user_item_feats_df = tst_user_item_feats_df.merge(user_info, on='user_id',how='left')


In [ ]:
trn_user_item_feats_df.columns


### 文章的特征直接读入

In [ ]:
articles =  pd.read_csv(data_path / 'articles.csv')
articles = reduce_mem(articles)


In [ ]:
# 拼上文章特征
trn_user_item_feats_df = trn_user_item_feats_df.merge(articles, left_on='click_article_id', right_on='article_id')

if val_user_item_feats_df is not None:
    val_user_item_feats_df = val_user_item_feats_df.merge(articles, left_on='click_article_id', right_on='article_id')
else:
    val_user_item_feats_df = None

tst_user_item_feats_df = tst_user_item_feats_df.merge(articles, left_on='click_article_id', right_on='article_id')


### 召回文章的主题是否在用户的爱好里面

In [ ]:
trn_user_item_feats_df['is_cat_hab'] = trn_user_item_feats_df.apply(lambda x: 1 if x.category_id in set(x.cate_list) else 0, axis=1)
if val_user_item_feats_df is not None:
    val_user_item_feats_df['is_cat_hab'] = val_user_item_feats_df.apply(lambda x: 1 if x.category_id in set(x.cate_list) else 0, axis=1)
else:
    val_user_item_feats_df = None
# TODO: 这里因为是sample数据原因，tst_user_item_feats_df 大小为0，当使用全量数据时，需要删除这行
if tst_user_item_feats_df.shape[0] > 0:
    tst_user_item_feats_df['is_cat_hab'] = tst_user_item_feats_df.apply(lambda x: 1 if x.category_id in set(x.cate_list) else 0, axis=1)


In [ ]:
# ============================================================================
# 📊 删除不需要的列
# ============================================================================

# 删除 cate_list 列（已用于构造 is_cat_hab 特征）
del trn_user_item_feats_df['cate_list']

if val_user_item_feats_df is not None:
    del val_user_item_feats_df['cate_list']
    
if tst_user_item_feats_df.shape[0] > 0:
    del tst_user_item_feats_df['cate_list']

# 删除 article_id 列（与 click_article_id 重复）
del trn_user_item_feats_df['article_id']

if val_user_item_feats_df is not None:
    del val_user_item_feats_df['article_id']
    
if tst_user_item_feats_df.shape[0] > 0:
    del tst_user_item_feats_df['article_id']

print("✅ 已删除不需要的列 (cate_list, article_id)")

## 保存特征

In [ ]:
# ============================================================================
# 📊 保存最终特征（包含用户特征和文章特征）
# ============================================================================

print("\n" + "=" * 70)
print("📂 保存最终排序特征")
print("=" * 70)

trn_user_item_feats_df.to_csv(save_path / 'trn_user_item_feats_df_all.csv', index=False)
print(f"   ✅ 训练特征数据: {len(trn_user_item_feats_df)} 条")

if val_user_item_feats_df is not None:
    val_user_item_feats_df.to_csv(save_path / 'val_user_item_feats_df_all.csv', index=False)
    print(f"   ✅ 验证特征数据: {len(val_user_item_feats_df)} 条")

tst_user_item_feats_df.to_csv(save_path / 'tst_user_item_feats_df_all.csv', index=False)
print(f"   ✅ 测试特征数据: {len(tst_user_item_feats_df)} 条")

mode_info = {
    'mode': MODE,
    'protocol': 'all_leave_last',
    'trn_users': len(trn_user_item_feats_df['user_id'].unique()) if 'user_id' in trn_user_item_feats_df.columns else 0,
    'val_users': len(val_user_item_feats_df['user_id'].unique()) if val_user_item_feats_df is not None else 0,
    'tst_users': len(tst_user_item_feats_df['user_id'].unique()) if 'user_id' in tst_user_item_feats_df.columns else 0,
}
pickle.dump(mode_info, open(save_path / 'feature_eng_mode_info_all.pkl', 'wb'))

print("\n" + "=" * 70)
print(f"🎉 特征工程完成！模式: {MODE}")
print("=" * 70)
print("📌 下一步：运行 jrc-ranking_all.ipynb 进行 JRC 排序训练和验证")
print("=" * 70)


## 总结

特征工程和数据清洗转换是比赛中至关重要的一块， 因为**数据和特征决定了机器学习的上限，而算法和模型只是逼近这个上限而已**，所以特征工程的好坏往往决定着最后的结果，**特征工程**可以一步增强数据的表达能力，通过构造新特征，我们可以挖掘出数据的更多信息，使得数据的表达能力进一步放大。 在本节内容中，我们主要是先通过制作特征和标签把预测问题转成了监督学习问题，然后围绕着用户画像和文章画像进行一系列特征的制作， 此外，为了保证正负样本的数据均衡，我们还学习了负采样就技术等。当然本节内容只是对构造特征提供了一些思路，也请学习者们在学习过程中开启头脑风暴，尝试更多的构造特征的方法，也欢迎我们一块探讨和交流。